In [ ]:
import numpy as np
import pandas as pd
from glob import glob
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, f1_score, precision_score, recall_score, ConfusionMatrixDisplay, accuracy_score
import time
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.discriminant_analysis import QuadraticDiscriminantAnalysis
from sklearn.neighbors import KNeighborsClassifier
from sklearn import svm
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

from sklearn.metrics import classification_report

import warnings
warnings.filterwarnings("ignore")

### Classes and dataset configuration

In [ ]:
#Features ordenation
dataset_dict = {
    'Dataset1': ["Bel-Pao","Copia","ERV","Gypsy","L1","Mariner","Non-TE","hAT"],
    'Dataset2': ["LINE","LTR","Non-TE","CLASS II"],
    'Dataset3': ["Bel-Pao","CACTA","Copia","ERV","Gypsy","L1","Mariner","Mutator","PIF","Non-TE","SINE","hAT"],
    'Dataset4': ["LINE","LTR","Non-TE", "SINE","CLASS II"],
    'Dataset5': ["LINE","LTR","Non-TE","CLASS II"]
}

# Desired order
dataset_dict_artigo = {
    'Dataset1': ["Copia","Gypsy","Bel-Pao","ERV","L1","Mariner","hAT","Non-TE"],
    'Dataset2': ["LTR","LINE","CLASS II","Non-TE"],
    'Dataset3': ["Copia","Gypsy","Bel-Pao","ERV","L1","SINE","Mariner","hAT","Mutator","PIF","CACTA","Non-TE"],
    'Dataset4': ["LTR","LINE", "SINE","CLASS II","Non-TE"],
    'Dataset5': ["LTR","LINE","CLASS II","Non-TE"]
}
# Mapping for datasets
dataset_dict_ord_artigo = {
    'Dataset1': [1,3,0,2,4,5,7,6],
    'Dataset2': [1,0,3,2],
    'Dataset3': [2,4,0,3,5,10,6,11,7,8,1,9],
    'Dataset4': [1,0, 3,4,2],
    'Dataset5': [1,0,3,2]
}

### Models to evaluate

In [ ]:
models = (
    KNeighborsClassifier(),
    RandomForestClassifier(random_state=42),
    GaussianNB(),
    svm.LinearSVC(random_state=42),
    DecisionTreeClassifier(random_state=42),
    AdaBoostClassifier(random_state=42),
    LinearDiscriminantAnalysis(),
)


# title for the plots
titles = (
    "KNN",
    "RF",
    "NaiveBayes",
    "SVMLinear",
    "DecisionTree",
    "AdaBoost",
    "LDA",

)

### Features in folder deep-features/terl to evaluate
features_arr = ['RELU6','RELU7','RELU6+RELU7']

### Functions to calculate metrics

In [ ]:
# Return the  accuracy by class
def accuracy_by_class(cm):
     # Initialize a list to store the accuracy of each class
    accuracies = []

    # NNumber of classes
    num_classes = cm.shape[0]

    for i in range(num_classes):
        # True positives (TP) for class i
        TP = cm[i, i]

        # True negatives (TN) for class i
        TN = np.sum(cm) - np.sum(cm[i, :]) - np.sum(cm[:, i]) + TP

        # False positives (FP) and False negatives (FN)
        FP = np.sum(cm[:, i]) - TP
        FN = np.sum(cm[i, :]) - TP

        # Calculation of accuracy for class i
        total = TP + TN + FP + FN
        if total > 0:
            accuracy = (TP + TN) / total
        else:
            accuracy = 0.0  # Case of no instances

        accuracies.append(accuracy)

    return accuracies

def format_by_class(metrics_by_class, labels ):
    return [f'{labels[i]}: {x:.3f}' for i,x in enumerate(metrics_by_class)]

def specificity(cm):
    # List to store the specificity of each class
    specifics = []

    # For each class
    for i in range(cm.shape[0]):
        # True negatives: sum of all other elements in the confusion matrix, except the row and column of the current class
        TN = np.sum(np.delete(np.delete(cm, i, axis=0), i, axis=1))

        # False positives: sum of the class column minus the true positives
        FP = np.sum(cm[:, i]) - cm[i, i]

        # Specificity = TN / (TN + FP)
        specificity_value = TN / (TN + FP)
        specifics.append(specificity_value)

    return specifics

#### DataFrame to store results

In [ ]:
dados = {
    "Dataset":[],
    "feature_arr":[],
    "Model": [],
    "Accuracy": [],
    "Precision": [],
    "Recall": [],
    "f1-score": [],
    "Specificity": [],
}

### Shows all results iterating over the datasets

In [ ]:
# Loop over all datasets
for dataset,labels in dataset_dict.items():
    # Path to features
    path = "./deep-features/terl/"+dataset+"/"

    # Labels Train and Test
    train_labels = np.load(path + "out_labels_train.npy")
    test_labels = np.load(path + "out_labels_test.npy")

    train_relu6 = np.load(path + "out_features_train_relu6.npy")
    train_relu7 = np.load(path + "out_features_train_relu7.npy")

    test_relu6 = np.load(path + "out_features_test_relu6.npy")
    test_relu7 = np.load(path + "out_features_test_relu7.npy")

    train_con = np.concatenate((train_relu6, train_relu7), axis=1)
    test_con = np.concatenate((test_relu6, test_relu7), axis=1)

    train_test_arr = [(train_relu6,test_relu6),(train_relu7,test_relu7),(train_con,test_con)]

    i = 0

    for X_train,X_test in train_test_arr:
        feat = features_arr[i]
        print("Experiment: "+dataset+" - "+feat)
        i += 1
        # Normalize the data
        X_train = X_train / np.max(X_train)
        X_test = X_test / np.max(X_test)
        #labels
        Y_train = train_labels
        Y_test = test_labels
        # Execute the estimators
        mdls = (clf.fit(X_train, Y_train) for clf in models)
        for clf, title in zip(mdls, titles):
            start = time.time()
            Y_pred = clf.predict(X_test)
            end = time.time()
            print("Time: ", end - start)
            print(title)
            report = classification_report(Y_test, Y_pred, target_names=labels,digits=3)
            dict_report = classification_report(Y_test, Y_pred, target_names=labels,digits=5,output_dict=True)
            print(report)
            cm = confusion_matrix(Y_test, Y_pred)
            acc_by_class = accuracy_by_class(cm)
            print("Accuracy by class: ", format_by_class(acc_by_class, labels))
            accuracy_macro = np.mean(acc_by_class)  # Mean of class accuracies
            print(f"Macro Accuracy: {accuracy_macro:.3f}")
            specificity_by_class = specificity(cm)
            print("Specificity by class: ", format_by_class(specificity_by_class,labels))
            specificity_macro = np.mean(specificity_by_class)
            print(f"Macro Specificity: {specificity_macro:.3f}")
            precision_macro = precision_score(Y_test, Y_pred, average='macro')
            recall_macro = recall_score(Y_test, Y_pred, average='macro')
            f1_score_macro = f1_score(Y_test, Y_pred, average='macro')
            # Populate data
            print(dict_report)
            dados["Dataset"].append(dataset)
            dados["Model"].append(title)
            dados["feature_arr"].append(feat)
            dados["Accuracy"].append(accuracy_macro)
            dados["Precision"].append(precision_macro)
            dados["Recall"].append(recall_macro)
            dados["f1-score"].append(f1_score_macro)
            dados["Specificity"].append(specificity_macro)
            print(dados)

            # Define the new order of labels and reorder the confusion matrix
            new_order = dataset_dict_ord_artigo[dataset]  # Desired order of classes
            cm_reordered = cm[new_order, :]  # Reorder rows
            cm_reordered = cm_reordered[:, new_order]  # Reorder columns

            disp = ConfusionMatrixDisplay(confusion_matrix=cm_reordered, display_labels=dataset_dict_artigo[dataset])
            fig, ax = plt.subplots(figsize=(10,10))
            disp.plot(ax=ax,cmap = 'Blues')
            plt.savefig("./results/"+dataset+"_"+features_arr[i-1]+"_"+title+".png")
            plt.show()
        print("------------------------------------------------------------------")

# Save the metrics
df = pd.DataFrame(dados)

for dataset in dataset_dict.keys():
    print("Analysis of dataset: "+dataset)
    df_dataset = df[df["Dataset"] == dataset]
    df_dataset.to_csv("./results/"+dataset+".csv",index=False)